In [ ]:
import os
import pandas as pd
import scipy.io
import scanpy as sc
from scipy import sparse
import numpy as np
import memento 
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.stats.multitest as smm

In [ ]:
##Defining Plotting :Significance
def label_significance(row):
    if row["de_pval_adj"] < fdr_thresh and row["de_coef"] > fc_thresh:
        return "Up-regulated"
    elif row["de_pval_adj"] < fdr_thresh and row["de_coef"] < -fc_thresh:
        return "Down-regulated"
    else:
        return "NS"  # not significant

In [ ]:
##Defining Plotting :VolcanoPlots
fdr_thresh = 0.05
fc_thresh  = 0.263
fig = "Bcells_RURAL_VolcanoPlot.png"

def volcano_up_down(df, fdr=fdr_thresh, fc_thresh=fc_thresh,
                    title="Volcano Plot: Up / Down / NS", fig=fig):

    plt.style.use("ggplot")

    df = df.copy()

    # Use raw p-value for volcano y-axis
    df["neglog10_p"] = -np.log10(df["de_pval"].replace(0, np.nextafter(0, 1)))

    x = df["de_coef"].values
    y = df["neglog10_p"].values

    # Determine significance categories using adjusted p-value
    sig_up   = (df["de_pval_adj"] < fdr) & (df["de_coef"] >  fc_thresh)
    sig_down = (df["de_pval_adj"] < fdr) & (df["de_coef"] < -fc_thresh)
    ns       = ~(sig_up | sig_down)

    plt.figure(figsize=(7,6))

    # Plot groups
    plt.scatter(x[ns],       y[ns],       color="gray", alpha=0.6, s=14, label="NS")
    plt.scatter(x[sig_up],   y[sig_up],   color="red",  alpha=0.85, s=18, label="Up-regulated")
    plt.scatter(x[sig_down], y[sig_down], color="blue", alpha=0.85, s=18, label="Down-regulated")

    # Threshold lines
    plt.axhline(-np.log10(fdr), ls="--", lw=1, color="black")
    plt.axvline(fc_thresh, ls="--", lw=1, color="black")
    plt.axvline(-fc_thresh, ls="--", lw=1, color="black")

    # Label only significant genes
    sig = df[sig_up | sig_down]

    for _, row in sig.iterrows():
        plt.text(
            row["de_coef"],
            row["neglog10_p"],
            row["gene"],
            fontsize=6,
            ha="left",
            va="bottom"
        )

    plt.xlabel("log2 Fold-change (de_coef)")
    plt.ylabel("−log10(p-value)")
    plt.title(title)

    plt.legend(frameon=True)
    plt.tight_layout()
    plt.savefig(fig, dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
new_directory_path = "/.../.../h5ad_files/"
os.chdir(new_directory_path)

X = scipy.io.mmread("Bcells_counts.mtx").tocsr()

genes = pd.read_csv("Bcells_genes.tsv", header=None)[0].astype(str).values
barcodes = pd.read_csv("Bcells_barcodes.tsv", header=None)[0].astype(str).values
obs = pd.read_csv("Bcells_metadata.csv", index_col=0)

# transpose so X becomes cells x genes
X = X.T.tocsr()

# align metadata rows to barcodes (and verify)
obs = obs.reindex(barcodes)

print(X.shape)        # should be (n_cells, n_genes)
print(obs.shape)      # should be (n_cells, n_meta_cols)
print(len(genes))     # should be n_genes
print(len(barcodes))  # should be n_cells

adata = sc.AnnData(X=X, obs=obs)
adata.var_names = genes
adata.obs_names = barcodes

adata.write_h5ad("Bcells.h5ad")

In [ ]:
new_directory_path = "/.../.../"
os.chdir(new_directory_path)

In [ ]:
adata = sc.read('Bcells.h5ad')
print(adata)

In [ ]:
adata.obs['Pool'].value_counts()

In [ ]:
capture_rate_dict = {
16: 0.1115,
4: 0.13825,
5: 0.1035,
6: 0.1015,
7: 0.1,
8: 0.11275,
9: 0.0905,
10: 0.075,
11: 0.07525,
12: 0.0895,
13: 0.129,
14: 0.11675,
15: 0.14,
1: 0.144,
2: 0.12625,
3: 0.11225,
}

In [ ]:
adata.obs["capture_rate"] = adata.obs["Pool"].astype(int).map(capture_rate_dict)

In [ ]:
adata.obs["capture_rate"].value_counts().plot(
    kind='pie', autopct='%1.1f%%', figsize=(5,5), ylabel=''
)
plt.title("capture_rate Distribution")
plt.savefig("Bcells_Stimulation_Distribution.png", dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
adata.obs[["Pool", "capture_rate"]].groupby("Pool").first()

In [ ]:
adata.obs['Stimulation'].value_counts().plot(
    kind='pie', autopct='%1.1f%%', figsize=(5,5), ylabel=''
)
plt.title("Stimulation Distribution")
plt.savefig("Bcells_Stimulation_Distribution.png", dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
##Split based on stimulation + sex
Unstim_Male = adata[
    (adata.obs["Stimulation"] == "Unstim") &
    (adata.obs["Sex"] == "Male")
].copy()
Unstim_Male.obs['Lifestyle'].value_counts().plot(
    kind='pie', autopct='%1.1f%%', figsize=(5,5), ylabel=''
)
plt.title("Lifestyle Distribution of Unstim Male")
plt.savefig("Unstim-Male_Bcells_Stimulation_Distribution.png", dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
Unstim_Male.var_names[:5]

In [ ]:
##Convert Stimulations to numeric Unstim =0 and others 1
Unstim_Male.obs['Lifestyle'] = Unstim_Male.obs['Lifestyle'].apply(lambda x: 0 if x == 'RURAL' else 1)

In [ ]:
Unstim_Male.obs['Lifestyle'].value_counts()

In [ ]:
# Donors per condition
ct = Unstim_Male.obs.groupby(['donor_id','Lifestyle']).size().unstack(fill_value=0)

#print(ct.head())

# How many paired vs unpaired?
n_paired = (ct[0] > 0) & (ct[1] > 0)
print("# paired Samples", int(n_paired.sum()), "# unpaired Samples" , int((~n_paired).sum()))

# Check covariate balance across conditions (per donor)
donor_meta = (Unstim_Male.obs
              .groupby('donor_id')[['Age','Sex']]
              .first())

donor_cond = ct.assign(has_ctrl=ct[0]>0, has_stim=ct[1]>0)
# e.g., compare Sex distribution for donors contributing to each condition
print ("\n# Samples in RURAL by Sex")
print(donor_meta.loc[donor_cond['has_ctrl']].Sex.value_counts())
print ("\n# Samples in URBAN by Sex")

print(donor_meta.loc[donor_cond['has_stim']].Sex.value_counts())

In [ ]:
Unstim_Male.obs['age_cov'] = Unstim_Male.obs['Age'].astype(float)
Unstim_Male.obs['donor_id'] = Unstim_Male.obs['donor_id'].astype(str)

In [ ]:
Unstim_Male.obs['capture_rate']

In [ ]:
memento.setup_memento(
    Unstim_Male,
    q_column='capture_rate',
)

In [ ]:
Cov_columns = ['Lifestyle', 'age_cov']
memento.create_groups(
    Unstim_Male,
    label_columns=Cov_columns
)
memento.compute_1d_moments(Unstim_Male)

In [ ]:
sample_meta = memento.get_groups(Unstim_Male)
print(sample_meta)
treatment_df  = sample_meta[['Lifestyle']].copy()
print(treatment_df)

In [ ]:
covariates_df = sample_meta[['age_cov']].copy()
covariates_df

In [ ]:
memento.ht_1d_moments(
    Unstim_Male,
    treatment=treatment_df,
    covariate=covariates_df,
    num_boot=5000,
    verbose=1,
    num_cpus=12)

In [ ]:
import os
print("Current path:", os.getcwd())

new_dir = "results_StimbySex_UrvsRu"
os.makedirs(new_dir, exist_ok=True)

# Move into that folder
os.chdir(new_dir)

print("Current path:", os.getcwd())

In [ ]:
Unstim_Male_result_cov = memento.get_1d_ht_result(Unstim_Male)
Unstim_Male_result_cov

In [ ]:
plt.scatter(Unstim_Male_result_cov.de_coef, Unstim_Male_result_cov.dv_coef, s=20)
plt.savefig("Bcells_Unstim_Male_ScatterPlot.png", dpi=300, bbox_inches='tight')

In [ ]:
from statsmodels.stats.multitest import multipletests

Unstim_Male_result_cov["de_pval_adj"] = multipletests(
    Unstim_Male_result_cov["de_pval"], alpha=0.05, method='fdr_bh'
)[1]

Unstim_Male_result_cov["dv_pval_adj"] = multipletests(
    Unstim_Male_result_cov["dv_pval"], alpha=0.05, method='fdr_bh'
)[1]


In [ ]:
Unstim_Male_result_cov

In [ ]:
fdr=0.05
fc_thresh=0.263
title="Unstim Male Volcano Plot: Up / Down / NS"
fig="Bcells_Unstim_Male_VolcanoPlot.png"


volcano_up_down(Unstim_Male_result_cov, fdr=fdr_thresh, fc_thresh=fc_thresh,fig=fig,title=title)

In [ ]:
Unstim_Male_result_cov["significance"] = Unstim_Male_result_cov.apply(label_significance, axis=1)
Unstim_Male_result_cov["significance"].value_counts()

In [ ]:
Unstim_Male_result_cov.to_excel("Bcells_Unstim_Male_Covar_AGE_Results.xlsx")

In [ ]:
Unstim_Male_result_cov

In [ ]:
## LPS RUN

In [ ]:
##Split based on stimulation + sex
LPS_Male = adata[
    (adata.obs["Stimulation"] == "LPS") &
    (adata.obs["Sex"] == "Male")
].copy()
LPS_Male.obs['Lifestyle'].value_counts().plot(
    kind='pie', autopct='%1.1f%%', figsize=(5,5), ylabel=''
)
plt.title("Lifestyle Distribution of LPS")
plt.savefig("LPS_Bcells_LPS_Male_Lifestyle-Distribution.png", dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
LPS_Male.var_names[:5]

In [ ]:
LPS_Male.obs['Lifestyle'] = LPS_Male.obs['Lifestyle'].apply(lambda x: 0 if x == 'RURAL' else 1)
LPS_Male.obs['Lifestyle'].value_counts()

In [ ]:
# Donors per condition
ct = LPS_Male.obs.groupby(['donor_id','Lifestyle']).size().unstack(fill_value=0)

#print(ct.head())

# How many paired vs unpaired?
n_paired = (ct[0] > 0) & (ct[1] > 0)
print("# paired Samples", int(n_paired.sum()), "# unpaired Samples" , int((~n_paired).sum()))

# Check covariate balance across conditions (per donor)
donor_meta = (LPS_Male.obs
              .groupby('donor_id')[['Age', 'Sex']]
              .first())

donor_cond = ct.assign(has_ctrl=ct[0]>0, has_stim=ct[1]>0)
# e.g., compare Sex distribution for donors contributing to each condition
print ("\n# Samples in RURAL by Sex")
print(donor_meta.loc[donor_cond['has_ctrl']].Sex.value_counts())
print ("\n# Samples in URBAN by Sex")

print(donor_meta.loc[donor_cond['has_stim']].Sex.value_counts())

In [ ]:
LPS_Male.obs['age_cov'] = LPS_Male.obs['Age'].astype(float)
LPS_Male.obs['donor_id'] = LPS_Male.obs['donor_id'].astype(str)

In [ ]:
memento.setup_memento(
    LPS_Male,
    q_column='capture_rate',
)

In [ ]:
Cov_columns = ['Lifestyle', 'age_cov']
memento.create_groups(
    LPS_Male,
    label_columns=Cov_columns
)

In [ ]:
memento.compute_1d_moments(LPS_Male)

In [ ]:
sample_meta = memento.get_groups(LPS_Male)
print(sample_meta)
treatment_df  = sample_meta[['Lifestyle']].copy()
print(treatment_df)
covariates_df = sample_meta[['age_cov']].copy()
print(covariates_df)

In [ ]:
memento.ht_1d_moments(
    LPS_Male,
    treatment=treatment_df,
    covariate=covariates_df,
    num_boot=5000,
    verbose=1,
    num_cpus=12
)

In [ ]:
LPS_Male_result_cov = memento.get_1d_ht_result(LPS_Male)
print(LPS_Male_result_cov)

In [ ]:
plt.scatter(LPS_Male_result_cov.de_coef, LPS_Male_result_cov.dv_coef, s=20)
plt.savefig("Bcells_LPSMale_ScatterPlot.png", dpi=300, bbox_inches='tight')

In [ ]:
from statsmodels.stats.multitest import multipletests
LPS_Male_result_cov["de_pval_adj"] = multipletests(
    LPS_Male_result_cov["de_pval"], alpha=0.05, method='fdr_bh'
)[1]
LPS_Male_result_cov["dv_pval_adj"] = multipletests(
    LPS_Male_result_cov["dv_pval"], alpha=0.05, method='fdr_bh'
)[1]

In [ ]:
fdr=0.05
fc_thresh=0.263
title="LPS Male Volcano Plot: Up / Down / NS"
fig="Bcells_LPS_Male_VolcanoPlot.png"
volcano_up_down(LPS_Male_result_cov, fdr=fdr_thresh, fc_thresh=fc_thresh,fig=fig,title=title)

In [ ]:
LPS_Male_result_cov["significance"] = LPS_Male_result_cov.apply(label_significance, axis=1)
print(LPS_Male_result_cov["significance"].value_counts())

LPS_Male_result_cov.to_excel("Bcells_LPS_Male_Covar_AGE_Results.xlsx")

In [ ]:
## Unstim Female

In [ ]:
##Split based on stimulation + sex
Unstim_Female = adata[
    (adata.obs["Stimulation"] == "Unstim") &
    (adata.obs["Sex"] == "Female")
].copy()
Unstim_Female.obs['Lifestyle'].value_counts().plot(
    kind='pie', autopct='%1.1f%%', figsize=(5,5), ylabel=''
)
plt.title("Lifestyle Distribution of Unstim Female")
plt.savefig("Unstim_Bcells_Unstim_Female_Lifestyle-Distribution.png", dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
Unstim_Female.var_names[:5]

In [ ]:
Unstim_Female.obs['Lifestyle'] = Unstim_Female.obs['Lifestyle'].apply(lambda x: 0 if x == 'RURAL' else 1)
Unstim_Female.obs['Lifestyle'].value_counts()

In [ ]:
# Donors per condition
ct = Unstim_Female.obs.groupby(['donor_id','Lifestyle']).size().unstack(fill_value=0)

#print(ct.head())

# How many paired vs unpaired?
n_paired = (ct[0] > 0) & (ct[1] > 0)
print("# paired Samples", int(n_paired.sum()), "# unpaired Samples" , int((~n_paired).sum()))

# Check covariate balance across conditions (per donor)
donor_meta = (Unstim_Female.obs
              .groupby('donor_id')[['Age', 'Sex']]
              .first())

donor_cond = ct.assign(has_ctrl=ct[0]>0, has_stim=ct[1]>0)
# e.g., compare Sex distribution for donors contributing to each condition
print ("\n# Samples in RURAL by Sex")
print(donor_meta.loc[donor_cond['has_ctrl']].Sex.value_counts())
print ("\n# Samples in URBAN by Sex")

print(donor_meta.loc[donor_cond['has_stim']].Sex.value_counts())

In [ ]:
Unstim_Female.obs['age_cov'] = Unstim_Female.obs['Age'].astype(float)
Unstim_Female.obs['donor_id'] = Unstim_Female.obs['donor_id'].astype(str)

In [ ]:
memento.setup_memento(
    Unstim_Female,
    q_column='capture_rate',
)

In [ ]:
Cov_columns = ['Lifestyle', 'age_cov']
memento.create_groups(
    Unstim_Female,
    label_columns=Cov_columns
)

In [ ]:
memento.compute_1d_moments(Unstim_Female)

In [ ]:
sample_meta = memento.get_groups(Unstim_Female)
print(sample_meta)
treatment_df  = sample_meta[['Lifestyle']].copy()
print(treatment_df)
covariates_df = sample_meta[['age_cov']].copy()
print(covariates_df)

In [ ]:
memento.ht_1d_moments(
    Unstim_Female,
    treatment=treatment_df,
    covariate=covariates_df,
    num_boot=5000,
    verbose=1,
    num_cpus=12
)

In [ ]:
Unstim_Female_result_cov = memento.get_1d_ht_result(Unstim_Female)
print(Unstim_Female_result_cov)

In [ ]:
plt.scatter(Unstim_Female_result_cov.de_coef, Unstim_Female_result_cov.dv_coef, s=20)
plt.savefig("Bcells_Unstim_Female_ScatterPlot.png", dpi=300, bbox_inches='tight')

In [ ]:
from statsmodels.stats.multitest import multipletests
Unstim_Female_result_cov["de_pval_adj"] = multipletests(
    Unstim_Female_result_cov["de_pval"], alpha=0.05, method='fdr_bh'
)[1]
Unstim_Female_result_cov["dv_pval_adj"] = multipletests(
    Unstim_Female_result_cov["dv_pval"], alpha=0.05, method='fdr_bh'
)[1]

In [ ]:
fdr=0.05
fc_thresh=0.263
title="Unstim Female Volcano Plot: Up / Down / NS"
fig="Bcells_Unstim_Female_VolcanoPlot.png"
volcano_up_down(Unstim_Female_result_cov, fdr=fdr_thresh, fc_thresh=fc_thresh,fig=fig,title=title)

In [ ]:
Unstim_Female_result_cov["significance"] = Unstim_Female_result_cov.apply(label_significance, axis=1)
print(Unstim_Female_result_cov["significance"].value_counts())

Unstim_Female_result_cov.to_excel("Bcells_Unstim_Female_Covar_AGE_Results.xlsx")

In [ ]:
## LPS Female

In [ ]:
##Split based on stimulation + sex
LPS_Female = adata[
    (adata.obs["Stimulation"] == "LPS") &
    (adata.obs["Sex"] == "Female")
].copy()
LPS_Female.obs['Lifestyle'].value_counts().plot(
    kind='pie', autopct='%1.1f%%', figsize=(5,5), ylabel=''
)
plt.title("Lifestyle Distribution of LPS Female")
plt.savefig("LPS_Bcells_LPS_Female_Lifestyle-Distribution.png", dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
LPS_Female.var_names[:5]

In [ ]:
LPS_Female.obs['Lifestyle'] = LPS_Female.obs['Lifestyle'].apply(lambda x: 0 if x == 'RURAL' else 1)
LPS_Female.obs['Lifestyle'].value_counts()

In [ ]:
# Donors per condition
ct = LPS_Female.obs.groupby(['donor_id','Lifestyle']).size().unstack(fill_value=0)

#print(ct.head())

# How many paired vs unpaired?
n_paired = (ct[0] > 0) & (ct[1] > 0)
print("# paired Samples", int(n_paired.sum()), "# unpaired Samples" , int((~n_paired).sum()))

# Check covariate balance across conditions (per donor)
donor_meta = (LPS_Female.obs
              .groupby('donor_id')[['Age', 'Sex']]
              .first())

donor_cond = ct.assign(has_ctrl=ct[0]>0, has_stim=ct[1]>0)
# e.g., compare Sex distribution for donors contributing to each condition
print ("\n# Samples in RURAL by Sex")
print(donor_meta.loc[donor_cond['has_ctrl']].Sex.value_counts())
print ("\n# Samples in URBAN by Sex")

print(donor_meta.loc[donor_cond['has_stim']].Sex.value_counts())

In [ ]:
LPS_Female.obs['age_cov'] = LPS_Female.obs['Age'].astype(float)
LPS_Female.obs['donor_id'] = LPS_Female.obs['donor_id'].astype(str)

In [ ]:
memento.setup_memento(
    LPS_Female,
    q_column='capture_rate',
)

In [ ]:
Cov_columns = ['Lifestyle', 'age_cov']
memento.create_groups(
    LPS_Female,
    label_columns=Cov_columns
)

In [ ]:
memento.compute_1d_moments(LPS_Female)

In [ ]:
sample_meta = memento.get_groups(LPS_Female)
print(sample_meta)
treatment_df  = sample_meta[['Lifestyle']].copy()
print(treatment_df)
covariates_df = sample_meta[['age_cov']].copy()
print(covariates_df)

In [ ]:
memento.ht_1d_moments(
    LPS_Female,
    treatment=treatment_df,
    covariate=covariates_df,
    num_boot=5000,
    verbose=1,
    num_cpus=12
)

In [ ]:
LPS_Female_result_cov = memento.get_1d_ht_result(LPS_Female)
print(LPS_Female_result_cov)

In [ ]:
plt.scatter(LPS_Female_result_cov.de_coef, LPS_Female_result_cov.dv_coef, s=20)
plt.savefig("Bcells_LPS_Female_ScatterPlot.png", dpi=300, bbox_inches='tight')

In [ ]:
from statsmodels.stats.multitest import multipletests
LPS_Female_result_cov["de_pval_adj"] = multipletests(
    LPS_Female_result_cov["de_pval"], alpha=0.05, method='fdr_bh'
)[1]
LPS_Female_result_cov["dv_pval_adj"] = multipletests(
    LPS_Female_result_cov["dv_pval"], alpha=0.05, method='fdr_bh'
)[1]

In [ ]:
fdr=0.05
fc_thresh=0.263
title="LPS Female Volcano Plot: Up / Down / NS"
fig="Bcells_LPS_Female_VolcanoPlot.png"
volcano_up_down(LPS_Female_result_cov, fdr=fdr_thresh, fc_thresh=fc_thresh,fig=fig,title=title)

In [ ]:
LPS_Female_result_cov["significance"] = LPS_Female_result_cov.apply(label_significance, axis=1)
print(LPS_Female_result_cov["significance"].value_counts())

LPS_Female_result_cov.to_excel("Bcells_LPS_Female_Covar_AGE_Results.xlsx")